# 读取文件

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 设置支持中文的字体（例如 SimHei），同时确保负号能正常显示
plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1) 读数据
df = pd.read_csv("../data/7.匹配时间/merged_output.csv")

print(df.head())

# 解析时间

In [ ]:
# 2) 解析时间字段
df["case_dt_hour"] = pd.to_datetime(df["case_dt_hour"], errors="coerce")
df = df.dropna(subset=["case_dt_hour"]).copy()

# ✅ 只保留 2020-12-31（含）及之前的数据
# df = df[df["case_dt_hour"] <= pd.Timestamp("2020-12-31 23:59:59")].copy()

# ✅ 只保留 2000-01-01（含）到 2020-12-31（含）的数据
start = pd.Timestamp("2000-01-01 00:00:00")
end   = pd.Timestamp("2020-12-31 23:59:59")
df = df[df["case_dt_hour"].between(start, end, inclusive="both")].copy()

# 3) 抽取 年/月/小时
df["year"] = df["case_dt_hour"].dt.year
df["month"] = df["case_dt_hour"].dt.month
df["hour"] = df["case_dt_hour"].dt.hour

# 4) 计数（补齐缺失的月份/小时，保证 1-12、0-23 都有柱子）
year_cnt  = df["year"].value_counts().sort_index()
month_cnt = df["month"].value_counts().sort_index().reindex(range(1, 13), fill_value=0)
hour_cnt  = df["hour"].value_counts().sort_index().reindex(range(0, 24), fill_value=0)

In [ ]:
year_cnt

# 绘图

In [ ]:
# -----------------------------
# (a) 小时分布
# -----------------------------
x = hour_cnt.index.tolist()
y = hour_cnt.values.tolist()

fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.bar(x, y, color="#7098d8")

ax.set_xlabel("Hour (0-23)")
ax.set_ylabel("Count")
ax.set_title("(a) Hour Distribution")
ax.grid(axis="y", linestyle="--", alpha=0.3)

for xi, yi in zip(x, y):
    if yi > 0:
        ax.text(xi, yi, f"{yi:,}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()

plt.savefig('../figure/fig3/time_hour.png', dpi=300,
            bbox_inches='tight', # 自动裁剪掉周围多余的白边
            pad_inches=0.1)       # 裁剪后保留一点点边距


plt.show()



In [ ]:
# -----------------------------
# (b) 月份分布（Jan. Feb. ...）
# -----------------------------
month_map = {
    1: "Jan.", 2: "Feb.", 3: "Mar.", 4: "Apr.", 5: "May.", 6: "Jun.",
    7: "Jul.", 8: "Aug.", 9: "Sep.", 10: "Oct.", 11: "Nov.", 12: "Dec."
}

x = month_cnt.index.tolist()
y = month_cnt.values.tolist()

fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.bar(x, y, color="#73afb2")

ax.set_xlabel("Month")
ax.set_ylabel("Count")
ax.set_title("(b) Month Distribution")
ax.grid(axis="y", linestyle="--", alpha=0.3)

for xi, yi in zip(x, y):
    if yi > 0:
        ax.text(xi, yi, f"{yi:,}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([month_map.get(m, str(m)) for m in x])  # 核心：把 1-12 显示成 Jan. Feb...
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig('../figure/fig3/time_month.png', dpi=300,
            bbox_inches='tight', # 自动裁剪掉周围多余的白边
            pad_inches=0.1)       # 裁剪后保留一点点边距

plt.show()


In [ ]:
# -----------------------------
# (c) 年份分布
# -----------------------------
x = year_cnt.index.tolist()
y = year_cnt.values.tolist()

fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.bar(x, y, color="#e6aeaf")

ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.set_title("(c) Year Distribution")
ax.grid(axis="y", linestyle="--", alpha=0.3)

for xi, yi in zip(x, y):
    if yi > 0:
        ax.text(xi, yi, f"{yi:,}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

plt.savefig('../figure/fig3/time_year.png', dpi=300,
            bbox_inches='tight', # 自动裁剪掉周围多余的白边
            pad_inches=0.1)       # 裁剪后保留一点点边距
plt.show()
